In [1]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

In [3]:
# Load your data
training_data = pd.read_parquet("UNSW_NB15_testing-set.parquet")
testing_data = pd.read_parquet("UNSW_NB15_training-set.parquet")


In [4]:
training_data


,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.000011,udp,-,INT,2,0,496,0,90909.093750,1.803636e+08,...,0,0,1,1,0,0,0,0,Normal,0
1,0.000008,udp,-,INT,2,0,1762,0,125000.000000,8.810000e+08,...,0,0,1,1,0,0,0,0,Normal,0
2,0.000005,udp,-,INT,2,0,1068,0,200000.000000,8.544000e+08,...,0,0,1,1,0,0,0,0,Normal,0
3,0.000006,udp,-,INT,2,0,900,0,166666.656250,6.000000e+08,...,0,0,2,1,0,0,0,0,Normal,0
4,0.000010,udp,-,INT,2,0,2126,0,100000.000000,8.504000e+08,...,0,0,2,1,0,0,0,0,Normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
82327,0.000005,udp,-,INT,2,0,104,0,200000.000000,8.320000e+07,...,0,0,1,1,0,0,0,0,Normal,0
82328,1.106101,tcp,-,FIN,20,8,18062,354,24.410067,1.241044e+05,...,0,0,1,1,0,0,0,0,Normal,0
82329,0.000000,arp,-,INT,1,0,46,0,0.000000,0.000000e+00,...,0,0,1,1,0,0,0,1,Normal,0
82330,0.000000,arp,-,INT,1,0,46,0,0.000000,0.000000e+00,...,0,0,1,1,0,0,0,1,Normal,0


In [5]:
testing_data

,dur,proto,service,state,spkts,dpkts,sbytes,dbytes,rate,sload,...,trans_depth,response_body_len,ct_src_dport_ltm,ct_dst_sport_ltm,is_ftp_login,ct_ftp_cmd,ct_flw_http_mthd,is_sm_ips_ports,attack_cat,label
0,0.121478,tcp,-,FIN,6,4,258,172,74.087486,1.415894e+04,...,0,0,1,1,0,0,0,0,Normal,0
1,0.649902,tcp,-,FIN,14,38,734,42014,78.473373,8.395112e+03,...,0,0,1,1,0,0,0,0,Normal,0
2,1.623129,tcp,-,FIN,8,16,364,13186,14.170161,1.572272e+03,...,0,0,1,1,0,0,0,0,Normal,0
3,1.681642,tcp,ftp,FIN,12,12,628,770,13.677108,2.740179e+03,...,0,0,1,1,1,1,0,0,Normal,0
4,0.449454,tcp,-,FIN,10,6,534,268,33.373825,8.561499e+03,...,0,0,2,1,0,0,0,0,Normal,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
175336,0.000009,udp,dns,INT,2,0,114,0,111111.109375,5.066666e+07,...,0,0,24,13,0,0,0,0,Generic,1
175337,0.505762,tcp,-,FIN,10,8,620,354,33.612648,8.826286e+03,...,0,0,1,1,0,0,0,0,Shellcode,1
175338,0.000009,udp,dns,INT,2,0,114,0,111111.109375,5.066666e+07,...,0,0,3,3,0,0,0,0,Generic,1
175339,0.000009,udp,dns,INT,2,0,114,0,111111.109375,5.066666e+07,...,0,0,30,14,0,0,0,0,Generic,1


In [6]:

# Preprocess categorical features using one-hot encoding
training_data_encoded = pd.get_dummies(training_data)
testing_data_encoded = pd.get_dummies(testing_data)

# Align the columns in training and testing data after one-hot encoding
X_train_encoded, X_test_encoded = training_data_encoded.align(testing_data_encoded, join='inner', axis=1)


In [7]:
# Convert to numpy arrays
                              #For testing on small data
# X_train = X_train_encoded.values[:150]
# X_test = X_test_encoded.values[:150]
# # Extract labels
# y_train = training_data['label'].values[:150]
# y_test = testing_data['label'].values[:150]
                            #For Whole data
X_train = X_train_encoded.values
X_test = X_test_encoded.values

# Extract labels
y_train = training_data['label'].values
y_test = testing_data['label'].values


In [10]:
# Define a fitness function based on classifier accuracy
def fitness(features):
    clf = RandomForestClassifier(n_estimators=100, random_state=42)
    clf.fit(X_train[:, features], y_train)
    y_pred = clf.predict(X_test[:, features])
    return accuracy_score(y_test, y_pred)

In [11]:
# Genetic Algorithm parameters
population_size = 50
num_generations = 10
mutation_rate = 0.1

# Initialize population randomly
population = np.random.randint(0, 2, size=(population_size, X_train.shape[1]))  # Assuming binary encoding for features

# Main loop for genetic algorithm
for generation in range(num_generations):
    # Evaluate fitness of each individual in the population
    fitness_scores = np.array([fitness(individual) for individual in population])

    # Select individuals for reproduction
    selected_indices = np.random.choice(range(population_size), size=population_size // 2, replace=False, p=fitness_scores / np.sum(fitness_scores))
    selected_individuals = population[selected_indices]


KeyboardInterrupt: 

In [ ]:
# Perform crossover
# Perform crossover
offspring = []
for i in range(0, len(selected_individuals) - 1, 2):
 parent1 = selected_individuals[i]
 parent2 = selected_individuals[i + 1]
 crossover_point = np.random.randint(1, len(parent1))
 child1 = np.concatenate((parent1[:crossover_point], parent2[crossover_point:]))
 child2 = np.concatenate((parent2[:crossover_point], parent1[crossover_point:]))
 offspring.extend([child1, child2])

 # Perform mutation
 for i in range(len(offspring)):
     for j in range(len(offspring[i])):
         if np.random.rand() < mutation_rate:
             offspring[i][j] = 1 - offspring[i][j]

 # Replace old population with new population
 population[:len(offspring)] = offspring

 # Print best individual in each generation
 best_individual_index = np.argmax(fitness_scores)
 best_individual = population[best_individual_index]
 print(f"Generation {generation+1}: Best Fitness = {fitness_scores[best_individual_index]}")


In [ ]:
# Get the best individual after all generations
best_individual_index = np.argmax(fitness_scores)
best_individual = population[best_individual_index]
selected_features = np.where(best_individual == 1)[0]

# Use the selected features to train a classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train[:, selected_features], y_train)
y_pred = clf.predict(X_test[:, selected_features])

In [ ]:
clf

In [ ]:
best_individual_index

In [ ]:
best_individual

In [ ]:
selected_features

In [ ]:
# Evaluate performance on the testing set
accuracy = accuracy_score(y_test, y_pred)
print(f"Final Test Accuracy: {accuracy}")